# Hybrid Search for Agentic Systems: BM25 + kNN

> Based on: [How to Build Agentic RAG with Hybrid Search — Towards Data Science](https://towardsdatascience.com/how-to-build-agentic-rag-with-hybrid-search/)

---

## What We'll Cover

1. **Why pure vector search fails** — the vocabulary gap problem  
2. **BM25 sparse retrieval** — keyword matching with TF-IDF+ scoring  
3. **kNN dense retrieval** — semantic embeddings and HNSW  
4. **Hybrid fusion** — Reciprocal Rank Fusion (RRF) and weighted convex combination  
5. **Agentic RAG loop** — turning retrieval into an LLM-callable tool  
6. **Evaluation** — Hit Rate, MRR, NDCG@K  

---

### The Core Problem

A standard RAG pipeline looks like this:

```
User query → [Vector DB: kNN search] → Top-K chunks → LLM → Answer
```

This breaks when a user asks:
- `"Error code ER_NO_DEFAULT_FOR_FIELD"` — exact error string
- `"What does SKU-4821 cost?"` — product identifier
- `"Find the section about §3.2 compliance"` — legal clause reference

Vector embeddings **average meaning across dimensions** — rare exact identifiers get diluted.  
BM25 handles these perfectly. Hybrid search gets the best of both worlds.

## 1. Setup & Dependencies

In [ ]:
%pip install -q rank_bm25 sentence-transformers faiss-cpu anthropic numpy scikit-learn

In [ ]:
import math
import json
import numpy as np
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Any

# BM25
from rank_bm25 import BM25Okapi

# Dense embeddings + FAISS ANN index
from sentence_transformers import SentenceTransformer
import faiss

print("All imports OK")

---
## 2. BM25 — Sparse Retrieval

### Theory

BM25 (*Best Match 25*) is the industry-standard keyword ranking algorithm. It scores every document $d$ against a query $q$ as:

$$\text{score}(d, q) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{f(t,d) \cdot (k_1 + 1)}{f(t,d) + k_1 \cdot \left(1 - b + b \cdot \frac{|d|}{\text{avgdl}}\right)}$$

| Symbol | Meaning |
|--------|---------|
| $f(t,d)$ | Term frequency of token $t$ in document $d$ |
| $\|d\|$ | Document length in tokens |
| avgdl | Average document length across corpus |
| $k_1$ | Term saturation (typically 1.2–2.0) |
| $b$ | Length normalization (typically 0.75) |
| IDF | $\log\frac{N - n_t + 0.5}{n_t + 0.5}$ — rewards rare terms |

**Key insight:** BM25 uses an **inverted index** (token → doc list). No GPU needed, sub-millisecond at scale.

### When BM25 wins
- Exact identifiers: error codes, SKUs, PINs, legal clause numbers  
- Rare technical terms that only appear in the target document  
- Queries where the user's vocabulary exactly matches the document's vocabulary

In [ ]:
# --- Sample corpus ---------------------------------------------------------
DOCS = [
    {"id": 0, "text": "BM25 uses an inverted index for fast keyword retrieval."},
    {"id": 1, "text": "Dense vector search finds semantically similar documents using embeddings."},
    {"id": 2, "text": "Error code ER_NO_DEFAULT_FOR_FIELD occurs in MySQL when inserting a row."},
    {"id": 3, "text": "Reciprocal Rank Fusion merges multiple ranked lists without score normalization."},
    {"id": 4, "text": "HNSW is an approximate nearest neighbour algorithm used in vector databases."},
    {"id": 5, "text": "Python database queries can be slow without proper indexing strategies."},
    {"id": 6, "text": "PostgreSQL optimization techniques include EXPLAIN ANALYZE and index hints."},
    {"id": 7, "text": "SKU-4821 is a premium standing desk with height-adjustable frame."},
    {"id": 8, "text": "Agentic RAG gives the language model a retrieval tool it can invoke iteratively."},
    {"id": 9, "text": "Hybrid search improves recall by 15-30% over single-method retrieval."},
]

# Tokenise (whitespace + lowercase — use a proper tokeniser in production)
tokenised_corpus = [doc["text"].lower().split() for doc in DOCS]

bm25 = BM25Okapi(tokenised_corpus)

def bm25_search(query: str, top_k: int = 5) -> list[tuple[int, float]]:
    tokens = query.lower().split()
    scores = bm25.get_scores(tokens)
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
    return [(doc_id, score) for doc_id, score in ranked[:top_k] if score > 0]

# Demo
query_kw = "ER_NO_DEFAULT_FOR_FIELD MySQL error"
results = bm25_search(query_kw)
print(f"BM25 results for: '{query_kw}'")
for doc_id, score in results:
    print(f"  [{doc_id:02d}] score={score:.3f}  |  {DOCS[doc_id]['text']}")

---
## 3. kNN Dense Retrieval

### Theory

Dense retrieval encodes both queries and documents into a **continuous vector space** using a transformer encoder (e.g. `all-MiniLM-L6-v2`, `text-embedding-3-small`).

At index time:
$$\mathbf{e}_d = \text{Encoder}(d) \in \mathbb{R}^{n}$$

At query time, find the $k$ nearest document vectors:
$$\text{sim}(\mathbf{e}_q, \mathbf{e}_d) = \frac{\mathbf{e}_q \cdot \mathbf{e}_d}{\|\mathbf{e}_q\| \cdot \|\mathbf{e}_d\|}$$

**HNSW (Hierarchical Navigable Small World)** is the standard ANN index — sub-linear lookup in $O(\log N)$ rather than brute-force $O(N)$.

### When dense retrieval wins
- Synonym bridging: `"slow database queries"` → finds `"PostgreSQL optimization techniques"`  
- Paraphrase matching: `"how to fix import errors"` → finds `"resolving module not found exceptions"`  
- Conversational, conceptual, or multi-hop questions

In [ ]:
# Build a FAISS flat index (exact cosine via inner-product on normalised vectors)
model = SentenceTransformer("all-MiniLM-L6-v2")

corpus_texts = [doc["text"] for doc in DOCS]
corpus_embeddings = model.encode(corpus_texts, normalize_embeddings=True)  # L2-normalised → dot == cosine

dim = corpus_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # Inner Product on unit vectors = cosine similarity
index.add(corpus_embeddings.astype("float32"))

print(f"FAISS index built: {index.ntotal} vectors of dim={dim}")


def dense_search(query: str, top_k: int = 5) -> list[tuple[int, float]]:
    q_emb = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(q_emb, top_k)
    return [(int(idx), float(score)) for idx, score in zip(indices[0], scores[0]) if idx != -1]


# Demo — semantic query with no exact keyword overlap
query_sem = "database performance optimisation"
results = dense_search(query_sem)
print(f"\nDense results for: '{query_sem}'")
for doc_id, score in results:
    print(f"  [{doc_id:02d}] score={score:.3f}  |  {DOCS[doc_id]['text']}")

---
## 4. Fusion Strategies

### 4a. Reciprocal Rank Fusion (RRF)

RRF is **score-agnostic** — it ignores raw scores and only uses rank positions. This sidesteps the scale mismatch between BM25 (unbounded floats) and cosine similarity (−1 to 1).

$$\text{RRF}(d) = \sum_{r \in \text{retrievers}} \frac{1}{k + \text{rank}_r(d)}$$

- $k = 60$ is the constant from the original 2009 paper (empirically robust)
- A document ranked **#1** scores $\frac{1}{61} \approx 0.0164$
- A document appearing in **both** lists scores the sum of both contributions

**Properties:**
- Zero labelled data required
- Robust to score distribution mismatches
- Best starting point for any hybrid system

### 4b. Convex Combination (Weighted Score Fusion)

$$\text{score}(d) = \alpha \cdot \hat{s}_{\text{dense}}(d) + (1 - \alpha) \cdot \hat{s}_{\text{sparse}}(d)$$

Both scores are min-max normalised to $[0, 1]$ first. $\alpha$ controls the balance:

| $\alpha$ | Behaviour |
|----------|-----------|
| 1.0 | Pure dense (semantic) |
| 0.7 | Semantic-heavy — good for conversational queries |
| 0.5 | Balanced |
| 0.3 | Keyword-heavy — good for technical docs / error codes |
| 0.0 | Pure BM25 |

Requires 50–100 labelled query pairs to tune $\alpha$ effectively.

In [ ]:
def reciprocal_rank_fusion(
    *ranked_lists: list[tuple[int, float]], k: int = 60
) -> list[tuple[int, float]]:
    """Merge ranked lists using RRF. Each list is [(doc_id, score), ...]."""
    rrf_scores: dict[int, float] = defaultdict(float)
    for ranked_list in ranked_lists:
        for rank, (doc_id, _score) in enumerate(ranked_list, start=1):
            rrf_scores[doc_id] += 1.0 / (k + rank)
    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)


def minmax_normalize(scores: list[tuple[int, float]]) -> list[tuple[int, float]]:
    if not scores:
        return scores
    values = [s for _, s in scores]
    lo, hi = min(values), max(values)
    if hi == lo:
        return [(doc_id, 1.0) for doc_id, _ in scores]
    return [(doc_id, (s - lo) / (hi - lo)) for doc_id, s in scores]


def convex_combination(
    dense_results: list[tuple[int, float]],
    sparse_results: list[tuple[int, float]],
    alpha: float = 0.5,
) -> list[tuple[int, float]]:
    """Weighted score fusion. alpha=1 → pure dense; alpha=0 → pure BM25."""
    dense_norm = dict(minmax_normalize(dense_results))
    sparse_norm = dict(minmax_normalize(sparse_results))
    all_ids = set(dense_norm) | set(sparse_norm)
    combined = {
        doc_id: alpha * dense_norm.get(doc_id, 0.0) + (1 - alpha) * sparse_norm.get(doc_id, 0.0)
        for doc_id in all_ids
    }
    return sorted(combined.items(), key=lambda x: x[1], reverse=True)


def hybrid_search(
    query: str, top_k: int = 5, fusion: str = "rrf", alpha: float = 0.5
) -> list[tuple[int, float]]:
    """Run BM25 + kNN then fuse. fusion='rrf' or 'convex'."""
    bm25_results = bm25_search(query, top_k=top_k * 2)
    dense_results = dense_search(query, top_k=top_k * 2)

    if fusion == "rrf":
        merged = reciprocal_rank_fusion(bm25_results, dense_results)
    else:
        merged = convex_combination(dense_results, bm25_results, alpha=alpha)

    return merged[:top_k]


def print_results(label: str, results: list[tuple[int, float]]) -> None:
    print(f"\n{'─'*60}")
    print(f"  {label}")
    print('─'*60)
    for doc_id, score in results:
        print(f"  [{doc_id:02d}] score={score:.4f}  |  {DOCS[doc_id]['text']}")

In [ ]:
# ── Compare all three strategies on two contrasting queries ─────────────────

queries = {
    "exact identifier query": "ER_NO_DEFAULT_FOR_FIELD MySQL error",
    "semantic / paraphrase query": "how to speed up database queries",
}

for label, q in queries.items():
    print(f"\n{'═'*60}")
    print(f"  QUERY ({label}): {q}")
    print_results("BM25 only", bm25_search(q, top_k=3))
    print_results("Dense (kNN) only", dense_search(q, top_k=3))
    print_results("Hybrid — RRF (k=60)", hybrid_search(q, top_k=3, fusion="rrf"))
    print_results("Hybrid — Convex α=0.5", hybrid_search(q, top_k=3, fusion="convex", alpha=0.5))

---
## 5. Agentic RAG with Hybrid Search

### From passive pipeline to active agent

**Passive RAG** — retrieval happens once, before the LLM:
```
User → retrieve(query) → [chunks] → LLM → answer
```

**Agentic RAG** — the LLM *decides* when and how to retrieve:
```
User → LLM (with tools) ─┬─ retrieve(query_1) → evaluate → ...
                          ├─ retrieve(query_2, alpha=0.2) → ...
                          └─ synthesise → answer
```

Benefits:
1. **Query rewriting** — the LLM rephrases the query before retrieval for better recall  
2. **Iterative retrieval** — if the first batch isn't enough, the agent retrieves again  
3. **Dynamic alpha selection** — the agent picks sparse/dense weight based on query type  
4. **Multi-hop reasoning** — retrieve → reason → retrieve again → answer  

### Implementation with Claude's tool use API

In [ ]:
import anthropic

# ── Tool definition exposed to Claude ───────────────────────────────────────
HYBRID_SEARCH_TOOL = {
    "name": "hybrid_search",
    "description": (
        "Search the document corpus using hybrid BM25 + kNN retrieval. "
        "Use alpha close to 0 for exact keyword queries (error codes, IDs), "
        "alpha close to 1 for semantic/conceptual queries, and alpha=0.5 for mixed queries."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query. Rephrase for better retrieval if needed.",
            },
            "top_k": {
                "type": "integer",
                "description": "Number of documents to return (default 3, max 5).",
                "default": 3,
            },
            "alpha": {
                "type": "number",
                "description": "Weight for dense (semantic) retrieval. 0=BM25 only, 1=kNN only, 0.5=balanced.",
                "default": 0.5,
            },
        },
        "required": ["query"],
    },
}


def execute_tool(tool_name: str, tool_input: dict) -> str:
    """Dispatch tool calls from the LLM to our local hybrid search."""
    if tool_name == "hybrid_search":
        query = tool_input["query"]
        top_k = min(int(tool_input.get("top_k", 3)), 5)
        alpha = float(tool_input.get("alpha", 0.5))
        results = hybrid_search(query, top_k=top_k, fusion="convex", alpha=alpha)
        hits = [
            {"doc_id": doc_id, "score": round(score, 4), "text": DOCS[doc_id]["text"]}
            for doc_id, score in results
        ]
        return json.dumps({"results": hits}, indent=2)
    return json.dumps({"error": f"Unknown tool: {tool_name}"})

In [ ]:
def agentic_rag(user_question: str, max_iterations: int = 5, verbose: bool = True) -> str:
    """
    Agentic RAG loop.
    
    The agent autonomously decides:
    - What query to issue (query rewriting)
    - What alpha to use (sparse vs dense weight)
    - Whether to retrieve again (iterative retrieval)
    """
    client = anthropic.Anthropic()

    system_prompt = (
        "You are a helpful assistant with access to a document corpus via the hybrid_search tool. "
        "Before answering, always search for relevant documents. "
        "If the initial results are insufficient, search again with a different query or alpha value. "
        "Choose alpha carefully: use low alpha (≈0.1) for exact keyword lookups, "
        "high alpha (≈0.9) for conceptual questions, and 0.5 for general queries."
    )

    messages = [{"role": "user", "content": user_question}]
    iteration = 0

    while iteration < max_iterations:
        iteration += 1
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=1024,
            system=system_prompt,
            tools=[HYBRID_SEARCH_TOOL],
            messages=messages,
        )

        if verbose:
            print(f"\n[Iteration {iteration}] stop_reason={response.stop_reason}")

        # Append assistant turn
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason == "end_turn":
            # Extract final text response
            for block in response.content:
                if hasattr(block, "text"):
                    return block.text
            return "(no text response)"

        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    if verbose:
                        print(f"  Tool call: {block.name}({json.dumps(block.input)})")
                    result = execute_tool(block.name, block.input)
                    if verbose:
                        print(f"  Tool result: {result[:300]}...")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result,
                    })
            messages.append({"role": "user", "content": tool_results})
        else:
            break

    return "(max iterations reached)"

In [ ]:
# ── Run the agent on two questions ──────────────────────────────────────────
# Set your ANTHROPIC_API_KEY env var before running this cell.
# export ANTHROPIC_API_KEY="sk-ant-..."

import os

if not os.environ.get("ANTHROPIC_API_KEY"):
    print("⚠  ANTHROPIC_API_KEY not set — skipping live API calls.\n"
          "Set it and re-run to see the full agentic loop.")
else:
    print("=" * 60)
    print("Question 1: Exact keyword lookup")
    print("=" * 60)
    answer1 = agentic_rag("What causes the ER_NO_DEFAULT_FOR_FIELD error?")
    print("\n── Final answer ──")
    print(answer1)

    print("\n" + "=" * 60)
    print("Question 2: Semantic / conceptual query")
    print("=" * 60)
    answer2 = agentic_rag("How can I make my database queries faster?")
    print("\n── Final answer ──")
    print(answer2)

---
## 6. Evaluation Metrics

Three standard metrics for retrieval systems. All assume a **relevance judgement set** — a list of `(query, relevant_doc_ids)` pairs.

| Metric | What it measures | Formula |
|--------|-----------------|---------|
| **Hit Rate @K** | Does any relevant doc appear in top-K? | $\frac{1}{Q}\sum_q \mathbf{1}[\text{rel} \cap \text{top-K} \neq \emptyset]$ |
| **MRR** | How high is the *first* relevant doc? | $\frac{1}{Q}\sum_q \frac{1}{\text{rank}_{\text{first relevant}}}$ |
| **NDCG@K** | Graded relevance — rewards earlier hits more | $\frac{\text{DCG@K}}{\text{IDCG@K}}$ |

Higher is better for all three. NDCG@K is the most informative for production systems.

In [ ]:
def hit_rate(results: list[tuple[int, float]], relevant_ids: set[int]) -> float:
    retrieved = {doc_id for doc_id, _ in results}
    return 1.0 if retrieved & relevant_ids else 0.0


def mrr(results: list[tuple[int, float]], relevant_ids: set[int]) -> float:
    for rank, (doc_id, _) in enumerate(results, start=1):
        if doc_id in relevant_ids:
            return 1.0 / rank
    return 0.0


def ndcg_at_k(results: list[tuple[int, float]], relevant_ids: set[int], k: int) -> float:
    dcg = sum(
        1.0 / math.log2(rank + 1)
        for rank, (doc_id, _) in enumerate(results[:k], start=1)
        if doc_id in relevant_ids
    )
    idcg = sum(1.0 / math.log2(rank + 1) for rank in range(1, min(len(relevant_ids), k) + 1))
    return dcg / idcg if idcg > 0 else 0.0


# ── Mini evaluation on our small corpus ─────────────────────────────────────
EVAL_SET = [
    {"query": "ER_NO_DEFAULT_FOR_FIELD MySQL error",    "relevant": {2}},
    {"query": "how to speed up database queries",       "relevant": {5, 6}},
    {"query": "approximate nearest neighbour vector DB","relevant": {4}},
    {"query": "fuse multiple ranked lists no scores",   "relevant": {3}},
    {"query": "agentic retrieval augmented generation", "relevant": {8, 9}},
]

strategies = {
    "BM25 only":      lambda q: bm25_search(q, top_k=5),
    "Dense only":     lambda q: dense_search(q, top_k=5),
    "Hybrid RRF":     lambda q: hybrid_search(q, top_k=5, fusion="rrf"),
    "Hybrid α=0.5":   lambda q: hybrid_search(q, top_k=5, fusion="convex", alpha=0.5),
}

print(f"{'Strategy':<20} {'Hit@3':>8} {'MRR':>8} {'NDCG@3':>8}")
print("─" * 48)

for name, retriever in strategies.items():
    hits, mrrs, ndcgs = [], [], []
    for item in EVAL_SET:
        res = retriever(item["query"])
        rel = item["relevant"]
        hits.append(hit_rate(res[:3], rel))
        mrrs.append(mrr(res, rel))
        ndcgs.append(ndcg_at_k(res, rel, k=3))
    print(f"{name:<20} {np.mean(hits):>8.3f} {np.mean(mrrs):>8.3f} {np.mean(ndcgs):>8.3f}")

---
## 7. Decision Guide — Which Strategy to Use?

```
                     ┌─────────────────────────────────────┐
                     │        What kind of query?           │
                     └──────────────┬──────────────────────┘
                                    │
              ┌─────────────────────┼─────────────────────┐
              ▼                     ▼                     ▼
        Exact identifiers    Mixed / general        Conceptual /
        (error codes,        queries                semantic
         SKUs, IDs)                                 (synonyms,
              │                     │                paraphrase)
              ▼                     ▼                     ▼
       Convex α ≈ 0.1        RRF (k=60)           Convex α ≈ 0.9
       or BM25 only          or Convex α=0.5      or Dense only
```

| Situation | Fusion | Alpha |
|-----------|--------|-------|
| Mixed queries, no labelled data | **RRF** k=60 | — |
| 50+ labelled pairs available | **Convex** | Tune via grid search |
| Technical docs, error codes, APIs | Convex | 0.2–0.3 |
| Conversational, support KB | Convex | 0.7–0.8 |
| Agentic system | Let the **LLM choose** alpha per query | Dynamic |

> **Empirical gains on BEIR benchmark:**  
> Hybrid vs dense-only: **+26–31% NDCG** on domains with high vocabulary mismatch  
> Hybrid vs BM25-only: **+18.5% MRR** on semantic / paraphrase-heavy domains

---
## 8. Production Checklist

```
□  Tokenisation        Use a proper tokeniser (NLTK, spaCy) — not just whitespace split
□  Stopword removal    Strip "the/a/is" before BM25 indexing
□  Chunking strategy   500–1000 token chunks with 10–20% overlap
□  Embedding model     Match embedding model to your domain (fine-tune if needed)
□  Index type          FAISS HNSW (local) / Qdrant / Weaviate / Elasticsearch (production)
□  Reranking           Add a cross-encoder reranker (e.g. ms-marco-MiniLM) on top-20 hits
□  Evaluation          Build a ≥100 query evaluation set before tuning alpha
□  Caching             Cache embeddings at index time; cache query embeddings for repeat queries
□  Monitoring          Log alpha values the agent picks — drift signals corpus change
```

## Sources

- [How to Build Agentic RAG with Hybrid Search — Towards Data Science](https://towardsdatascience.com/how-to-build-agentic-rag-with-hybrid-search/)
- [Hybrid Search for RAG: BM25, SPLADE, and Vector Search Combined — PremAI Blog](https://blog.premai.io/hybrid-search-for-rag-bm25-splade-and-vector-search-combined/)
- [Hybrid Search Explained — Weaviate](https://weaviate.io/blog/hybrid-search-explained)
- [Introducing Reciprocal Rank Fusion — OpenSearch](https://opensearch.org/blog/introducing-reciprocal-rank-fusion-hybrid-search/)
- [Full-text search for RAG apps: BM25 & hybrid search — Redis](https://redis.io/blog/full-text-search-for-rag-the-precision-layer/)